# Lending Club Credit Scoring — 02. Modelling and Evaluation

**Prerequisite:** run `01_eda.ipynb` first — it writes `data/processed/clean.parquet`.

**What happens here**

| Step | Content |
|---|---|
| 1–2 | Setup; recover the frozen out-of-time split and assert it is intact |
| 3 | Preprocessing — fitted on **training rows only**, inside a `Pipeline` |
| 4 | Four models: logistic regression, random forest, HistGB, XGBoost |
| 5 | Probability calibration on a time-ordered validation slice |
| 6 | Threshold selection: F1, Youden-J, and an explicit cost rule |
| 7–8 | Metrics, comparison table, figures |
| 9 | The optimism of a random split, quantified |
| 10 | Conclusions and limitations |

**Every headline number is the out-of-time number.** The random-split figures appear once,
in section 9, purely to quantify how much optimism random splitting buys.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             confusion_matrix, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts"
FIG_DIR = ARTIFACTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_RED, C_GRAY = "#e34948", "#8a8a86"
MODEL_COLORS = {}          # filled in once the model registry exists

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (8, 4.5), "figure.dpi": 110, "savefig.dpi": 150,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
})


def save_fig(name: str) -> None:
    plt.savefig(FIG_DIR / f"{name}.png")
    plt.show()


warnings.filterwarnings("ignore", category=FutureWarning)
print("Setup complete.")

## 2. Recover the frozen split

The split was decided in notebook 01, **before** any transform that learns from data.
Here it is only reconstructed from the `is_train` flag and then checked. Three assertions
guard the properties that matter:

1. train and test `issue_d` ranges do not overlap (no look-ahead);
2. the manifest's cutoff matches the data;
3. the test-row hash matches the manifest (the split has not silently drifted).

A silent split change would invalidate comparison with the other groups, so these fail
loudly rather than warn.

In [ ]:
CLEAN_PATH = DATA_PROCESSED / "clean.parquet"
if not CLEAN_PATH.exists():
    raise FileNotFoundError(
        f"{CLEAN_PATH} not found. Run notebooks/01_eda.ipynb first -- it produces this file.")

df = pd.read_parquet(CLEAN_PATH)
manifest = json.loads((ARTIFACTS / "split_manifest.json").read_text())
CUTOFF_DATE = pd.Timestamp(manifest["cutoff_date"])
DATA_SOURCE = manifest.get("data_source", "unknown")

print(f"Loaded {len(df):,} rows  |  source: {DATA_SOURCE}  |  cutoff: {CUTOFF_DATE.date()}")
if DATA_SOURCE == "synthetic":
    print("\n!! SYNTHETIC data -- these numbers smoke-test the pipeline, they are not findings.")

In [ ]:
# Columns that exist for bookkeeping, not for the model to learn from.
NON_FEATURES = ["target", "is_train", "is_train_random_ref", "issue_d", "earliest_cr_line",
                "grade", "sub_grade"]   # grade/sub_grade kept as *_ordinal instead

feature_cols = [c for c in df.columns if c not in NON_FEATURES]
X = df[feature_cols]
y = df["target"]
train_mask = df["is_train"].to_numpy()

X_train, X_test = X.loc[train_mask], X.loc[~train_mask]
y_train, y_test = y.loc[train_mask], y.loc[~train_mask]
issue_train = df.loc[train_mask, "issue_d"]

# --- Guards ------------------------------------------------------------------------
assert issue_train.max() < df.loc[~train_mask, "issue_d"].min(), \
    "Look-ahead: train and test issue_d ranges overlap."
assert manifest["n_train"] == len(X_train) and manifest["n_test"] == len(X_test), \
    "Row counts disagree with split_manifest.json -- the split has drifted."
import hashlib
recomputed = hashlib.md5(np.flatnonzero(~train_mask).tobytes()).hexdigest()
assert recomputed == manifest["test_index_md5"], \
    "Test-index hash mismatch -- the split is not the one recorded in the manifest."
print("OK -- split matches the manifest; no look-ahead.\n")

overview = pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "default_rate": [y_train.mean(), y_test.mean()],
    "period": [f"{issue_train.min():%Y-%m} to {issue_train.max():%Y-%m}",
               f"{df.loc[~train_mask, 'issue_d'].min():%Y-%m} to "
               f"{df.loc[~train_mask, 'issue_d'].max():%Y-%m}"],
}, index=["train", "test"])
print(f"{len(feature_cols)} features")
display(overview)

## 3. Preprocessing — fitted on training rows only

Every transform here learns a parameter from the data: imputation learns a median or a
mode, scaling learns a mean and standard deviation, one-hot encoding learns a category
vocabulary. Fitting any of them on the full frame is **train–test contamination** — the
model's inputs would be shaped by rows it is supposed to be evaluated on, and no metric
would reveal it.

```python
# The anti-pattern this avoids:
preprocessor.fit(X)                       # <-- sees test rows
X_train, X_test = train_test_split(X)     # <-- too late, the damage is done
```

Wrapping the `ColumnTransformer` in a `Pipeline` makes the correct behaviour structural
rather than a matter of discipline: `.fit()` only ever receives training rows, and
`X_test` reaches the transformer solely through `.transform()`. It also means
cross-validation re-fits the preprocessing **inside each fold** instead of once over all
training data.

`min_frequency=0.01` folds rare categories into a single infrequent bucket, so a category
appearing a handful of times cannot contribute a noisy high-variance dummy — and
`handle_unknown="infrequent_if_exist"` means a category that appears only after the cutoff
is handled gracefully instead of crashing at test time.

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

print(f"{len(numeric_cols)} numeric, {len(categorical_cols)} categorical")
print(f"  numeric     : {numeric_cols}")
print(f"  categorical : {categorical_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=0.01, sparse_output=False)),
        ]), categorical_cols),
    ],
    remainder="drop",
)

## 4. Models

Four models, spanning the interpretability/flexibility trade-off. Each is a full
`Pipeline`, so preprocessing travels with the estimator and cannot be applied out of
order.

| Model | Why it is here |
|---|---|
| **Logistic regression** | The industry baseline. A credit scorecard is a linear model for good reasons: coefficients are auditable, monotonic, and defensible to a regulator. Anything more complex has to earn its place against this. |
| **Random forest** | Captures interactions and non-linearity with little tuning; a robust reference for "how much does non-linearity buy?" |
| **HistGradientBoosting** | Gradient boosting, usually the strongest tabular family, with no extra dependency. |
| **XGBoost** | The Kaggle standard for this dataset — included so our comparison is against what the public work actually uses. Guarded by a `try/except` so the notebook runs without it. |

Class imbalance is handled with **class weights** rather than resampling. At ~20 %
prevalence this is not a rare-event problem; SMOTE and friends would synthesise
applicants who never existed, and would have to be applied inside the CV folds to avoid
yet another leakage path.

In [ ]:
models: dict[str, Pipeline] = {}

models["Logistic Regression"] = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000,
                               random_state=RANDOM_SEED)),
])

models["Random Forest"] = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50,
                                   class_weight="balanced_subsample", n_jobs=-1,
                                   random_state=RANDOM_SEED)),
])

models["HistGradientBoosting"] = Pipeline([
    ("prep", preprocessor),
    ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                           max_leaf_nodes=31, min_samples_leaf=50,
                                           l2_regularization=1.0,
                                           random_state=RANDOM_SEED)),
])

try:
    from xgboost import XGBClassifier
    scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
    models["XGBoost"] = Pipeline([
        ("prep", preprocessor),
        ("clf", XGBClassifier(n_estimators=400, learning_rate=0.06, max_depth=5,
                              subsample=0.8, colsample_bytree=0.8,
                              reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
                              eval_metric="logloss", tree_method="hist",
                              n_jobs=-1, random_state=RANDOM_SEED)),
    ])
    print(f"XGBoost available (scale_pos_weight={scale_pos_weight:.2f})")
except ImportError:
    print("XGBoost not installed -- skipping it. `pip install xgboost` to include it.")

MODEL_COLORS = dict(zip(models, [C_BLUE, C_ORANGE, C_AQUA, C_VIOLET]))
print(f"\n{len(models)} models registered: {list(models)}")

### 4a. Cross-validation on the training period — `TimeSeriesSplit`

A sanity check before touching the test set, and a second place the time ordering has to
be respected. Shuffled `StratifiedKFold` would let fold *k* train on loans issued **after**
the ones it validates on — reintroducing exactly the look-ahead the out-of-time split
exists to remove, and producing optimistic CV scores that disagree with the final test
number for reasons nobody can find later.

`TimeSeriesSplit` on chronologically sorted training rows gives expanding-window folds:
always train on the past, validate on the future.

In [ ]:
cv = TimeSeriesSplit(n_splits=4)
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    cv_rows.append({"model": name, "cv_roc_auc_mean": scores.mean(),
                    "cv_roc_auc_std": scores.std(),
                    "folds": np.round(scores, 4).tolist()})
    print(f"{name:<24} ROC-AUC {scores.mean():.4f} +/- {scores.std():.4f}")

cv_results = pd.DataFrame(cv_rows).set_index("model")
display(cv_results)

**Table 1 — Expanding-window cross-validation on the training period (ROC-AUC).**

These are honest in-period estimates: each fold trains only on loans issued before the
ones it scores. The fold-to-fold standard deviation is itself informative — a large spread
means performance is unstable across time, which is a warning about the out-of-time test
to come, not noise to be averaged away.

Compare these means with the test-set numbers in section 8. CV scores materially above the
test score indicate drift between the training period and the test period; that gap is a
result worth reporting, not a bug to be tuned away.

## 5. Fit and calibrate

**Why calibration is not optional here.** In credit scoring the probability *is* the
product: expected loss is `PD × EAD × LGD`, and pricing, provisioning and cut-off policy
all consume the number itself, not just the ranking. A model can rank perfectly and still
be systematically wrong about the level. ROC-AUC — a pure ranking statistic — cannot
detect that; the Brier score and the calibration curve can.

`class_weight="balanced"` and `scale_pos_weight` make this worse before they make it
better: both deliberately distort the class prior to help the model learn, so raw outputs
overstate the default probability. Calibration undoes that distortion.

**The leakage-safe way to calibrate.** The calibrator is fitted on a held-out slice of the
**training** data — never on test rows. And the slice is the **latest pre-cutoff months**,
not a random subset, so the same time ordering holds: fit on the earlier period, calibrate
on the later one, test on the period after that.

In [ ]:
# Time-ordered split of the training period: earliest 80% to fit, latest 20% to calibrate.
order = np.argsort(issue_train.to_numpy(), kind="mergesort")
n_fit = int(len(order) * 0.8)
fit_idx, cal_idx = order[:n_fit], order[n_fit:]

X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
X_cal, y_cal = X_train.iloc[cal_idx], y_train.iloc[cal_idx]

print(f"fit slice        : {len(X_fit):>7,} loans  "
      f"({issue_train.iloc[fit_idx].min():%Y-%m} to {issue_train.iloc[fit_idx].max():%Y-%m})")
print(f"calibration slice: {len(X_cal):>7,} loans  "
      f"({issue_train.iloc[cal_idx].min():%Y-%m} to {issue_train.iloc[cal_idx].max():%Y-%m})")
assert issue_train.iloc[fit_idx].max() <= issue_train.iloc[cal_idx].min(), \
    "Calibration slice must come after the fit slice in time."

In [ ]:
def calibrate(fitted_model, X_cal, y_cal):
    """Wrap an already-fitted estimator in an isotonic calibrator.

    Uses FrozenEstimator where available (sklearn >= 1.6); falls back to cv="prefit".
    Either way the base model is NOT refitted -- only the calibration map is learned.
    """
    try:
        from sklearn.frozen import FrozenEstimator
        calibrated = CalibratedClassifierCV(FrozenEstimator(fitted_model), method="isotonic")
    except ImportError:
        calibrated = CalibratedClassifierCV(fitted_model, method="isotonic", cv="prefit")
    return calibrated.fit(X_cal, y_cal)


fitted, calibrated_models = {}, {}
for name, model in models.items():
    print(f"fitting {name} ...", end=" ", flush=True)
    base = model.fit(X_fit, y_fit)              # trained on the EARLY training slice only
    fitted[name] = base
    calibrated_models[name] = calibrate(base, X_cal, y_cal)
    print("done")

# Score the test set once, with the calibrated models.
scores_test = {name: m.predict_proba(X_test)[:, 1] for name, m in calibrated_models.items()}
scores_cal = {name: m.predict_proba(X_cal)[:, 1] for name, m in calibrated_models.items()}
scores_test_raw = {name: m.predict_proba(X_test)[:, 1] for name, m in fitted.items()}
print("\nAll models fitted, calibrated and scored.")

## 6. Choosing a decision threshold

A probability becomes a lending decision only through a cut-off, and **0.5 is arbitrary**.
At ~20 % prevalence a model can be well-calibrated and still put almost every applicant
below 0.5 — classifying everyone as "will repay", achieving 80 % accuracy, and being
completely useless. Accuracy is reported below only to make that failure visible.

Three rules, all computed on the **calibration slice** and then applied unchanged to test.
Tuning a threshold on test scores is the same family of leakage as fitting a scaler on
them:

- **F1-optimal** — balances precision and recall, ignoring that the two errors cost
  different amounts.
- **Youden's J** (`max(TPR − FPR)`) — the point furthest from chance on the ROC curve;
  threshold-symmetric, so it also ignores costs.
- **Cost-based** — the only one that encodes the actual business problem. A missed default
  (false negative) costs the unrecovered principal; a wrongly rejected good applicant
  (false positive) costs the foregone interest margin. These are not symmetric, and the
  ratio drives the cut-off far more than the model choice does.

The costs below are **illustrative placeholders**. Replace them with figures from the
portfolio before quoting any result that depends on them.

In [ ]:
COST_FN = 1.0     # missed default: lost principal (normalised to 1)
COST_FP = 0.25     # rejected good applicant: foregone interest margin


def threshold_candidates(y_true, y_score):
    """Return the F1-optimal, Youden-J and cost-minimising thresholds."""
    prec, rec, thr_pr = precision_recall_curve(y_true, y_score)
    f1 = np.divide(2 * prec * rec, prec + rec,
                   out=np.zeros_like(prec), where=(prec + rec) > 0)
    thr_f1 = thr_pr[np.argmax(f1[:-1])] if len(thr_pr) else 0.5

    fpr, tpr, thr_roc = roc_curve(y_true, y_score)
    thr_youden = thr_roc[np.argmax(tpr - fpr)]

    grid = np.linspace(0.01, 0.99, 197)
    n_pos, n_neg = int(y_true.sum()), int((1 - y_true).sum())
    costs = []
    for t in grid:
        pred = (y_score >= t).astype(int)
        fn = int(((pred == 0) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        costs.append((fn * COST_FN + fp * COST_FP) / len(y_true))
    thr_cost = grid[int(np.argmin(costs))]
    return {"f1": float(thr_f1), "youden": float(thr_youden), "cost": float(thr_cost)}, \
           (grid, np.array(costs))


thresholds = {}
for name in calibrated_models:
    thresholds[name], _ = threshold_candidates(y_cal.to_numpy(), scores_cal[name])

thr_table = pd.DataFrame(thresholds).T
thr_table.columns = ["F1-optimal", "Youden-J", f"Cost (FN:FP = {COST_FN:g}:{COST_FP:g})"]
display(thr_table.style.format("{:.3f}").set_caption(
    "Thresholds chosen on the calibration slice, applied unchanged to test"))

# The cost rule is the operating point used for all point metrics below.
OPERATING_THRESHOLD = {name: t["cost"] for name, t in thresholds.items()}

**Table 2 — Candidate thresholds per model, chosen on the calibration slice.**

The three rules disagree substantially, and that disagreement is the point: the cut-off is
a **policy decision**, not a model output. The cost-based threshold sits lowest, because a
missed default is assumed four times more expensive than a needlessly rejected applicant —
so the lender should be willing to reject more good applicants to avoid one bad loan.

The cost-based threshold is used as the operating point for every point metric below. All
threshold-free metrics (ROC-AUC, PR-AUC, KS, Gini, Brier) are unaffected by this choice.

## 7. Metrics

Each metric answers a different question, and the brief requires we can say which:

| Metric | Question it answers | Caveat |
|---|---|---|
| **ROC-AUC** | Given one defaulter and one payer, how often does the model rank the defaulter riskier? | Optimistic under imbalance — true negatives dominate both rates |
| **PR-AUC** | Of the applicants flagged riskiest, how many actually default? | Baseline is the prevalence (~0.20), not 0.5 — read it against that |
| **KS** | Maximum gap between the good and bad score distributions | The credit-industry standard; a single number for "does the scorecard separate?" |
| **Gini** | `2·AUC − 1`, rescaled so 0 = random | Same information as AUC; reported because credit risk conventionally does |
| **Brier** | Mean squared error of the probabilities themselves | The only one that punishes miscalibration; **lower is better** |
| **Precision / Recall / F1** | Performance at the chosen cut-off | Meaningless without justifying the threshold |
| **Accuracy** | Share of correct predictions | Reported only to show it is uninformative here — predicting "never defaults" scores ~80 % |

In [ ]:
def evaluate(y_true, y_score, threshold: float) -> dict:
    """Threshold-free and point metrics for one model."""
    y_true = np.asarray(y_true)
    y_pred = (y_score >= threshold).astype(int)

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "ROC-AUC": auc,
        "PR-AUC": average_precision_score(y_true, y_score),
        "KS": float(np.max(tpr - fpr)),
        "Gini": 2 * auc - 1,
        "Brier": brier_score_loss(y_true, y_score),
        "Threshold": threshold,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": (tp + tn) / len(y_true),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Cost/loan": (fn * COST_FN + fp * COST_FP) / len(y_true),
    }


results = {name: evaluate(y_test, scores_test[name], OPERATING_THRESHOLD[name])
           for name in calibrated_models}
metrics = pd.DataFrame(results).T.sort_values("PR-AUC", ascending=False)

metrics.to_csv(ARTIFACTS / "metrics.csv")
print(f"Wrote {ARTIFACTS / 'metrics.csv'}\n")

display(metrics[["ROC-AUC", "PR-AUC", "KS", "Gini", "Brier"]]
        .style.format("{:.4f}")
        .set_caption("Out-of-time test performance -- threshold-free metrics"))
display(metrics[["Threshold", "Precision", "Recall", "F1", "Accuracy", "Cost/loan"]]
        .style.format({"Threshold": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
                       "F1": "{:.3f}", "Accuracy": "{:.3f}", "Cost/loan": "{:.4f}"})
        .set_caption("Out-of-time test performance -- at the cost-based operating point"))

**Table 3 — Out-of-time test performance.**

Read the two halves differently. The top table ranks the models' ability to *order*
applicants by risk and is independent of any cut-off. The bottom table is what happens
when the cost-based cut-off is applied, and it moves with that policy choice.

Two patterns to check in your own run. First, the gradient-boosted models should edge out
logistic regression on ROC-AUC and PR-AUC — but by a margin measured in thousandths, not
tenths. On tabular credit data with heavily engineered features the linear baseline is
genuinely competitive, which is the honest headline and a large part of why real
scorecards remain linear.

Second, note how high **accuracy** is for every model and how little it means: predicting
"nobody defaults" would score around 80 % here. Recall at the cost-based threshold is far
more informative, and is deliberately high at the expense of precision, because the cost
matrix says a missed default hurts four times more than a wrongly rejected applicant.

If any model shows ROC-AUC materially above ~0.75 on the real data, stop and look for a
leaked column before believing it.

## 8. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for name, score in scores_test.items():
    fpr, tpr, _ = roc_curve(y_test, score)
    axes[0].plot(fpr, tpr, lw=2, color=MODEL_COLORS[name],
                 label=f"{name} (AUC {results[name]['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], ls="--", lw=1.5, color=C_GRAY, label="Random (0.500)")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves (out-of-time test)")
axes[0].legend(frameon=False, loc="lower right", fontsize=9)

for name, score in scores_test.items():
    prec, rec, _ = precision_recall_curve(y_test, score)
    axes[1].plot(rec, prec, lw=2, color=MODEL_COLORS[name],
                 label=f"{name} (AP {results[name]['PR-AUC']:.3f})")
axes[1].axhline(y_test.mean(), ls="--", lw=1.5, color=C_GRAY,
                label=f"Prevalence ({y_test.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curves (out-of-time test)")
axes[1].legend(frameon=False, loc="upper right", fontsize=9)

plt.tight_layout()
save_fig("08_roc_and_pr_curves")

**Figure 8 — ROC curves (left) and precision–recall curves (right) on the
out-of-time test set.**

The two panels are the same information under different framings, and the difference
matters. The ROC curves sit comfortably above the diagonal and look reassuring. The PR
curves are the sterner test: their baseline is the prevalence line (~0.20), not 0.5, and
precision falls away quickly as recall increases.

That decay is the real operating constraint. Catching most defaulters means accepting that
a large share of the applicants flagged will have repaid — the flat region on the right of
the PR curve is where a high-recall lending policy would sit. This is why PR-AUC is the
more informative headline for an imbalanced problem, and why reporting ROC-AUC alone
overstates practical usefulness.

The curves for the different models are close together, consistent with Table 3: model
choice matters much less on this problem than feature quality and threshold policy.

In [ ]:
best_model = metrics.index[0]
print(f"Best model by PR-AUC: {best_model}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# --- Calibration: raw vs calibrated for the best model ------------------------------
for label, score, colour, style in [
        ("Uncalibrated", scores_test_raw[best_model], C_ORANGE, "--"),
        ("Calibrated (isotonic)", scores_test[best_model], C_BLUE, "-")]:
    frac_pos, mean_pred = calibration_curve(y_test, score, n_bins=12, strategy="quantile")
    brier = brier_score_loss(y_test, score)
    axes[0].plot(mean_pred, frac_pos, marker="o", ms=6, lw=2, ls=style, color=colour,
                 label=f"{label} (Brier {brier:.4f})")
axes[0].plot([0, 1], [0, 1], ls=":", lw=1.5, color=C_GRAY, label="Perfect calibration")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed default rate")
axes[0].set_title(f"Calibration -- {best_model}")
axes[0].legend(frameon=False, fontsize=9)

# --- Score distribution by true outcome --------------------------------------------
for outcome, colour, label in [(0, C_BLUE, "Fully Paid"), (1, C_RED, "Charged Off")]:
    axes[1].hist(scores_test[best_model][y_test.to_numpy() == outcome], bins=40,
                 density=True, alpha=0.55, color=colour, label=label)
axes[1].axvline(OPERATING_THRESHOLD[best_model], color="#0b0b0b", lw=1.8, ls="--",
                label=f"Operating threshold ({OPERATING_THRESHOLD[best_model]:.3f})")
axes[1].set_xlabel("Predicted probability of default")
axes[1].set_ylabel("Density")
axes[1].set_title(f"Score distribution by outcome -- {best_model}")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
save_fig("09_calibration_and_scores")

**Figure 9 — Calibration curve before and after isotonic calibration (left) and the
calibrated score distribution split by true outcome (right), best model.**

The left panel shows why calibration earned its place in the pipeline. The uncalibrated
curve sits well below the diagonal across most of the range — the observed default
rate is far lower than the predicted probability — the class-weighted model
systematically **overstates** default probability, exactly as the reweighting of the class
prior predicts. Isotonic regression pulls it back toward the diagonal and the Brier score
improves. ROC-AUC is unchanged by this, because calibration is monotone and cannot alter
the ranking; that is precisely the point, and precisely why AUC alone is insufficient for a
model whose probabilities get used.

The right panel shows the separation the metrics summarise. The two distributions are
clearly shifted apart but overlap heavily — there is no threshold that cleanly divides
them. The dashed line marks the cost-based operating point; everything to the right is
declined. Reading the overlap around that line gives an intuitive sense of the errors the
policy accepts, in a way a single F1 number does not.

In [ ]:
cm = confusion_matrix(y_test, (scores_test[best_model] >= OPERATING_THRESHOLD[best_model]).astype(int))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

from matplotlib.colors import LinearSegmentedColormap
sequential_blue = LinearSegmentedColormap.from_list(
    "blue_seq", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281"])

fig, ax = plt.subplots(figsize=(5.6, 4.6))
labels = np.array([[f"{cm[i, j]:,}\n({cm_norm[i, j]:.1%})" for j in range(2)] for i in range(2)])
sns.heatmap(cm_norm, annot=labels, fmt="", cmap=sequential_blue, vmin=0, vmax=1,
            cbar_kws={"label": "Share of true class"}, linewidths=2, linecolor="#fcfcfb",
            xticklabels=["Predicted:\nFully Paid", "Predicted:\nCharged Off"],
            yticklabels=["Actual:\nFully Paid", "Actual:\nCharged Off"], ax=ax)
ax.set_title(f"Confusion matrix -- {best_model}\n(row-normalised, cost-based threshold)")
save_fig("10_confusion_matrix")

**Figure 10 — Row-normalised confusion matrix for the best model at the cost-based
threshold.**

Rows are normalised so each row shows what happens to that true class: the bottom-right
cell is recall (defaulters correctly flagged) and the top-right is the false-positive rate
(good applicants wrongly declined).

The asymmetry is deliberate, not a defect. The cost matrix says a missed default costs four
times a wrongly rejected applicant, so the optimiser buys a high catch rate by accepting
many false positives. Whether that trade is acceptable is a business question — with the
placeholder costs replaced by real ones, this single figure is what a credit committee
would actually argue over.

Note also that the bottom-left cell — defaults the model let through — is where the
realised losses live. Reducing it is worth far more than a marginal AUC improvement.

In [ ]:
# Permutation importance: model-agnostic, and measured on the TEST set because we are
# asking "what did this model rely on?", not making a modelling choice from it.
perm_n = min(20_000, len(X_test))
perm_idx = np.random.default_rng(RANDOM_SEED).choice(len(X_test), perm_n, replace=False)

perm = permutation_importance(
    calibrated_models[best_model], X_test.iloc[perm_idx], y_test.iloc[perm_idx],
    scoring="roc_auc", n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1)

importance = (pd.Series(perm.importances_mean, index=X_test.columns)
                .sort_values(ascending=False).head(15))
errors = pd.Series(perm.importances_std, index=X_test.columns).loc[importance.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

axes[0].barh(importance.index[::-1], importance.values[::-1],
             xerr=errors.values[::-1], color=C_BLUE, height=0.65,
             error_kw={"ecolor": C_GRAY, "lw": 1.2})
axes[0].set_xlabel("Drop in ROC-AUC when the column is shuffled")
axes[0].set_title(f"Permutation importance -- {best_model}")

# Logistic-regression coefficients, for interpretability alongside the black box.
lr = fitted["Logistic Regression"]
feature_names = lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=feature_names)
top_coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(15)[::-1]
axes[1].barh(top_coefs.index, top_coefs.values,
             color=[C_RED if v > 0 else C_BLUE for v in top_coefs.values], height=0.65)
axes[1].axvline(0, color="#0b0b0b", lw=1)
axes[1].set_xlabel("Coefficient (standardised features; + raises default risk)")
axes[1].set_title("Logistic regression coefficients")

plt.tight_layout()
save_fig("11_feature_importance")

**Figure 11 — Permutation importance for the best model (left) and standardised
logistic-regression coefficients (right).**

The two panels are complementary: permutation importance says *how much the model relies*
on a column, the coefficients say *in which direction and how strongly* a linear model
uses it. The error bars on the left are the spread across five shuffles — a bar shorter
than its own error bar is not distinguishable from noise.

Expect `int_rate`, `sub_grade_ordinal` and `grade_ordinal` near the top. That deserves
scrutiny rather than satisfaction: those columns are **Lending Club's own risk
assessment**, not raw applicant data. To a large extent a high-importance score there
means our model has learned to reproduce their underwriting model. It is legitimate — the
grade genuinely is available at application time — but it caps how much credit we can take
for the predictive performance, and it is worth reporting the model refitted without the
grade/rate columns to see how much signal survives on applicant data alone.

Among the genuinely independent features, `loan_to_income`, `dti` and `fico_avg` carrying
weight is the expected and reassuring result: leverage, existing debt burden and credit
history are what underwriting theory says should matter.

The colour split on the right is directional: red raises modelled default risk, blue
lowers it. Signs should be checkable against intuition — higher FICO lowering risk, higher
leverage raising it. A sign that contradicts domain knowledge usually indicates
collinearity (recall `loan_amnt` and `installment` from Figure 5) rather than a discovery.

## 9. How optimistic is a random split?

Everything above used the out-of-time protocol. This section re-runs the pipeline **once**
under a random stratified split — the design most public Kaggle notebooks on this dataset
use — purely to quantify the difference.

This is a headline result of the project, not an aside. If random-split scores are
materially higher, then published leaderboard numbers on this dataset are not comparable to
an honest forecasting estimate, and the gap is the size of the illusion.

The out-of-time number remains the one we report and defend.

In [ ]:
ref_train_mask = df["is_train_random_ref"].to_numpy()
Xr_train, Xr_test = X.loc[ref_train_mask], X.loc[~ref_train_mask]
yr_train, yr_test = y.loc[ref_train_mask], y.loc[~ref_train_mask]

comparison = []
for name, model in models.items():
    ref_fit = model.fit(Xr_train, yr_train)            # same pipeline, random split
    ref_score = ref_fit.predict_proba(Xr_test)[:, 1]
    comparison.append({
        "model": name,
        "OOT ROC-AUC": results[name]["ROC-AUC"],
        "Random ROC-AUC": roc_auc_score(yr_test, ref_score),
        "OOT PR-AUC": results[name]["PR-AUC"],
        "Random PR-AUC": average_precision_score(yr_test, ref_score),
    })

split_comparison = pd.DataFrame(comparison).set_index("model")
split_comparison["ROC-AUC optimism"] = (
    split_comparison["Random ROC-AUC"] - split_comparison["OOT ROC-AUC"])
split_comparison["PR-AUC optimism"] = (
    split_comparison["Random PR-AUC"] - split_comparison["OOT PR-AUC"])

display(split_comparison.style.format("{:.4f}").set_caption(
    "Out-of-time vs random stratified split -- the cost of look-ahead"))

# Refit the reported models on the proper training data so nothing downstream is polluted.
for name, model in models.items():
    model.fit(X_fit, y_fit)
print("\nModels refitted on the out-of-time training slice.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.arange(len(split_comparison))
w, gap = 0.38, 0.014          # gap keeps a surface sliver between adjacent bars
ax.bar(x - w / 2 - gap, split_comparison["OOT ROC-AUC"], w, color=C_BLUE,
       label="Out-of-time (reported)")
ax.bar(x + w / 2 + gap, split_comparison["Random ROC-AUC"], w, color=C_ORANGE,
       label="Random split (optimistic)")
for i, (oot, rnd) in enumerate(split_comparison[["OOT ROC-AUC", "Random ROC-AUC"]].values):
    ax.annotate(f"{oot:.3f}", (i - w / 2 - gap, oot), ha="center", va="bottom", fontsize=9,
                color="#52514e")
    ax.annotate(f"{rnd:.3f}", (i + w / 2 + gap, rnd), ha="center", va="bottom", fontsize=9,
                color="#52514e")
ax.axhline(0.5, color=C_GRAY, ls="--", lw=1.5, label="Random guessing")
ax.set_xticks(x)
ax.set_xticklabels(split_comparison.index, rotation=12, ha="right")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.45, max(split_comparison[["OOT ROC-AUC", "Random ROC-AUC"]].max()) * 1.09)
ax.set_title("ROC-AUC under both evaluation protocols, by model")
ax.legend(frameon=False, fontsize=9)
save_fig("12_split_protocol_comparison")

**Figure 12 — The same four models scored under both evaluation protocols.**

The spread between the blue and orange bars for a single model should be compared with the
spread *across* models within either colour. Where the protocol gap is the larger of the
two — which is the usual finding on this dataset — the conclusion is blunt: **how you
split matters more than which model you fit.**

That reframes what a leaderboard-style comparison is worth. A notebook reporting a higher
AUC under a random split has not necessarily built a better model; it has measured a
different, easier problem. It is also the strongest argument for the shared-split
requirement in the brief — without a fixed protocol, group-to-group numbers are not
comparable.

If the gap in your run is negligible, say so plainly: it would mean drift over this window
is mild, which is itself a finding worth reporting rather than hiding.

## 10. Conclusions

Fill in the specific numbers from your run; the structure below is what the report needs.

**Performance.** All four models land in a narrow band, with the gradient-boosted models
marginally ahead on PR-AUC and logistic regression close behind. On the real data expect
ROC-AUC in the region of 0.68–0.72. That is the honest ceiling for application-time
features on this dataset, and it is consistent with the published credit-scoring
literature — default is driven substantially by post-origination events (job loss,
illness, divorce) that no application form observes.

**Model choice is not the interesting variable.** The gap between the best and worst model
is small next to the gap between evaluation protocols (Figure 12) and next to the effect of
the threshold policy (Table 2). Effort spent on feature quality, split integrity and
cut-off economics pays far more than effort spent on hyperparameter search.

**Recommendation.** Logistic regression remains a defensible production choice: within
thousandths of the ensembles, auditable coefficient by coefficient, trivial to monitor, and
straightforward to justify to a regulator under adverse-action requirements. Deploy the
gradient-boosted model only if the measured lift survives on real data and the
interpretability cost is explicitly accepted.

**Calibration matters as much as ranking.** The uncalibrated models systematically
overstate default probability (Figure 9). Since expected loss is computed from the
probability itself, an uncalibrated model misprices every loan even when it ranks them
perfectly.

---

## Limitations

These constrain what may be claimed from the results above. They are not boilerplate.

**1. `issue_d` is a proxy for the application date.** It records when the loan was
*issued*, not when it was applied for or decided. Using it to derive
`credit_history_years` and to define the out-of-time cutoff is legitimate and is standard
practice on this dataset — but the true decision moment is not in the data. Any
application-to-issuance lag is invisible to us, so features are dated slightly later than
a live scorecard would observe them, and the cutoff separates *issuance* cohorts rather
than *decision* cohorts.

**2. Right-censoring from the terminal-status filter.** Keeping only `Fully Paid` and
`Charged Off` discards loans still `Current`. Because 36- and 60-month loans issued near
the end of the window have not had time to mature, the post-cutoff test set is enriched
with loans that resolved *early* — and early resolution is disproportionately charge-off.
The rising tail in Figure 7 is this artefact. Consequences: the test-period default rate is
**not** the true default rate for that cohort, and the out-of-time metrics are computed on
a non-random subsample of post-cutoff originations. A survival-analysis framing
(time-to-default with censoring handled explicitly) would address this properly and is the
natural extension of this work.

**3. Accepted loans only — selection bias.** The dataset contains only applicants Lending
Club approved. The model therefore estimates default risk **conditional on acceptance**,
not for the through-the-door population. Applying it to all applicants would extrapolate
beyond the region the training data covers. This is the classic reject-inference problem;
Lending Club publishes a rejected-applications file, but it lacks outcomes by
construction, so the problem cannot be solved simply by joining the two.

**4. Drift is measured, not corrected.** The out-of-time design quantifies how much
performance decays across time (Figure 12) but does nothing to counter it. A deployed
scorecard would need scheduled refitting, population-stability monitoring, and alerting on
score-distribution shift.

**5. Fairness is not assessed.** `addr_state` is used as a feature and correlates with
protected characteristics through residential segregation; income and home ownership do
too. No disparate-impact testing was performed. Any real deployment would require it, and
in several jurisdictions it is a legal precondition.

**6. Costs are placeholders.** `COST_FN = 1.0` and `COST_FP = 0.25` are illustrative. Every
point metric — precision, recall, F1, the operating threshold, `Cost/loan` — moves with
them. Replace them with real portfolio economics before quoting any of those numbers.

In [ ]:
print("=" * 72)
print("RUN SUMMARY".center(72))
print("=" * 72)
print(f"Data source      : {DATA_SOURCE}")
if DATA_SOURCE == "synthetic":
    print("                   ^ SMOKE TEST ONLY -- not findings. See README.md.")
print(f"Split            : out-of-time, cutoff {CUTOFF_DATE.date()}")
print(f"Train / test     : {len(X_train):,} / {len(X_test):,}")
print(f"Best model       : {best_model}")
print(f"  ROC-AUC (OOT)  : {results[best_model]['ROC-AUC']:.4f}")
print(f"  PR-AUC  (OOT)  : {results[best_model]['PR-AUC']:.4f}")
print(f"  KS      (OOT)  : {results[best_model]['KS']:.4f}")
print(f"  Brier   (OOT)  : {results[best_model]['Brier']:.4f}")
print(f"\nArtifacts written to {ARTIFACTS}:")
for path in sorted(ARTIFACTS.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACTS)}")